# 01 — Validate IBGE input data

This notebook performs a **diagnostic-only validation** of the two GeoParquet inputs used in the Petrópolis OpenStreetMap study:

1. `Petropolis_2025_4674_IBGE.parquet` — municipal boundary;
2. `petropolis_grade_id26_4674_IBGE.parquet` — IBGE 200 m statistical grid.

The notebook does **not** aggregate the grid to 1 km and does not modify source files. It inspects Parquet/GeoParquet metadata, validates CRS and geometries, detects the official IBGE hierarchy fields, checks the uniqueness of the 200 m identifier, counts 200 m cells per official 1 km parent, validates metric dimensions, and produces a PASS/WARNING/FAIL summary for Notebook 02.

All code, variable names, comments, and outputs are in English.

In [ ]:
from pathlib import Path
import json
import re
import sys
import platform

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import pyarrow as pa
import pyarrow.parquet as pq

pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 160)
pd.set_option('display.max_colwidth', 120)

print('Python:', sys.version.split()[0])
print('Platform:', platform.platform())
print('GeoPandas:', gpd.__version__)
print('Pandas:', pd.__version__)
print('PyArrow:', pa.__version__)

## 1. Locate and validate the input files

In [ ]:
def find_project_root(start: Path) -> Path:
    """Find the repository root by locating the Petropolis directory."""
    start = start.resolve()
    for candidate in [start, start.parent, *start.parents]:
        if (candidate / 'Petropolis').exists():
            return candidate
    raise FileNotFoundError("Could not locate the repository root containing 'Petropolis'.")

PROJECT_ROOT = find_project_root(Path.cwd())
DATA_DIR = PROJECT_ROOT / 'Petropolis'
MUNICIPALITY_FILE = DATA_DIR / 'Petropolis_2025_4674_IBGE.parquet'
GRID_FILE = DATA_DIR / 'petropolis_grade_id26_4674_IBGE.parquet'

for path in [MUNICIPALITY_FILE, GRID_FILE]:
    if not path.exists():
        raise FileNotFoundError(f'Required input file not found: {path}')

print('Project root:', PROJECT_ROOT)
print('PASS — Both input files exist.')

## 2. Inspect physical Parquet and GeoParquet metadata

In [ ]:
def inspect_parquet_metadata(path: Path) -> dict:
    """Return Parquet schema and decoded GeoParquet metadata."""
    pf = pq.ParquetFile(path)
    raw = pf.metadata.metadata or {}
    decoded = {}
    for key, value in raw.items():
        k = key.decode('utf-8', errors='replace')
        try:
            text = value.decode('utf-8')
            decoded[k] = json.loads(text) if k == 'geo' else text
        except Exception:
            decoded[k] = '<binary or non-UTF8 metadata>'
    return {
        'rows': pf.metadata.num_rows,
        'row_groups': pf.metadata.num_row_groups,
        'columns': pf.metadata.num_columns,
        'schema': pf.schema_arrow,
        'metadata': decoded,
    }

municipality_meta = inspect_parquet_metadata(MUNICIPALITY_FILE)
grid_meta = inspect_parquet_metadata(GRID_FILE)

for label, meta in [('Municipality', municipality_meta), ('IBGE grid', grid_meta)]:
    print(f'\n--- {label} ---')
    print('Rows:', f"{meta['rows']:,}")
    print('Row groups:', meta['row_groups'])
    print('Columns:', meta['columns'])
    print(meta['schema'])
    print('Geo metadata:')
    print(json.dumps(meta['metadata'].get('geo'), indent=2, ensure_ascii=False))

## 3. Load datasets and inspect exact schemas

In [ ]:
municipality = gpd.read_parquet(MUNICIPALITY_FILE)
grid_200m = gpd.read_parquet(GRID_FILE)

def schema_table(gdf: gpd.GeoDataFrame) -> pd.DataFrame:
    return pd.DataFrame({
        'column': gdf.columns,
        'dtype': [str(dtype) for dtype in gdf.dtypes],
        'non_null': [int(gdf[col].notna().sum()) for col in gdf.columns],
        'null': [int(gdf[col].isna().sum()) for col in gdf.columns],
    })

print(f'Municipality features: {len(municipality):,}')
display(schema_table(municipality))
print(f'IBGE grid features: {len(grid_200m):,}')
display(schema_table(grid_200m))
display(grid_200m.head())

## 4. CRS validation
Both datasets are expected to use **SIRGAS 2000 / EPSG:4674**. Metric measurements are performed later in SIRGAS 2000 / UTM zone 23S (EPSG:31983).

In [ ]:
def epsg_or_none(gdf):
    return None if gdf.crs is None else gdf.crs.to_epsg()

municipality_epsg = epsg_or_none(municipality)
grid_epsg = epsg_or_none(grid_200m)

print('Municipality CRS:', municipality.crs)
print('Municipality EPSG:', municipality_epsg)
print('Grid CRS:', grid_200m.crs)
print('Grid EPSG:', grid_epsg)

display(pd.Series({
    'municipality_has_crs': municipality.crs is not None,
    'grid_has_crs': grid_200m.crs is not None,
    'same_crs': municipality.crs == grid_200m.crs,
    'municipality_epsg_4674': municipality_epsg == 4674,
    'grid_epsg_4674': grid_epsg == 4674,
}, name='result'))

## 5. Geometry diagnostics

In [ ]:
def geometry_diagnostics(gdf, label):
    geom = gdf.geometry
    non_empty = geom[geom.notna() & ~geom.is_empty]
    return {
        'dataset': label,
        'features': len(gdf),
        'geometry_column': gdf.geometry.name,
        'geometry_types': geom.geom_type.value_counts(dropna=False).to_dict(),
        'null_geometries': int(geom.isna().sum()),
        'empty_geometries': int(geom.is_empty.sum()),
        'invalid_geometries': int((~non_empty.is_valid).sum()),
        'duplicate_geometries': int(geom.duplicated().sum()),
        'bounds': tuple(gdf.total_bounds.tolist()),
    }

municipality_geom_diag = geometry_diagnostics(municipality, 'Municipality')
grid_geom_diag = geometry_diagnostics(grid_200m, 'IBGE 200 m grid')
display(pd.DataFrame([municipality_geom_diag, grid_geom_diag]))

## 6. Detect the official IBGE hierarchy fields
The aggregation in Notebook 02 will use the official `nome_1km` hierarchy rather than a newly generated geometric index.

In [ ]:
def find_column_case_insensitive(columns, candidates):
    lookup = {str(col).lower(): col for col in columns}
    for candidate in candidates:
        if candidate.lower() in lookup:
            return lookup[candidate.lower()]
    return None

ID_200M_COL = find_column_case_insensitive(grid_200m.columns, ['ID_UNICO', 'id_unico'])
PARENT_1KM_COL = find_column_case_insensitive(grid_200m.columns, ['nome_1km'])

hierarchy_patterns = {
    '200m_unique_id': ['ID_UNICO'],
    '1km_parent': ['nome_1km'],
    '5km_parent': ['nome_5km'],
    '10km_parent': ['nome_10km'],
    '50km_parent': ['nome_50km'],
    '100km_parent': ['nome_100km'],
    '500km_parent': ['nome_500km'],
    'quadrant': ['quadrante'],
}
detected_hierarchy = {role: find_column_case_insensitive(grid_200m.columns, names) for role, names in hierarchy_patterns.items()}
display(pd.DataFrame(detected_hierarchy.items(), columns=['hierarchy_role', 'detected_column']))

if ID_200M_COL is None:
    raise KeyError('Could not detect the IBGE 200 m unique-cell identifier field.')
if PARENT_1KM_COL is None:
    raise KeyError('Could not detect the IBGE 1 km parent-cell identifier field.')

## 7. Validate 200 m identifiers and 1 km parent identifiers

In [ ]:
id_series = grid_200m[ID_200M_COL]
parent_series = grid_200m[PARENT_1KM_COL]

display(pd.Series({
    '200m_id_non_null': bool(id_series.notna().all()),
    '200m_id_unique': bool(id_series.is_unique),
    '200m_duplicate_count': int(id_series.duplicated().sum()),
    '200m_unique_count': int(id_series.nunique(dropna=True)),
    '1km_parent_null_count': int(parent_series.isna().sum()),
    '1km_parent_unique_count': int(parent_series.nunique(dropna=True)),
}, name='value'))

print('Sample 200 m identifiers:')
display(id_series.dropna().astype(str).head(20).to_frame())
print('Sample 1 km parent identifiers:')
display(parent_series.dropna().astype(str).drop_duplicates().head(20).to_frame())

pattern_200m = re.compile(r'^200M[Ee]?\d+[Nn]?\d+$')
pattern_matches = id_series.dropna().astype(str).str.match(pattern_200m)
print('IDs matching expected textual pattern:', f'{pattern_matches.sum():,} / {len(pattern_matches):,}')

## 8. Count 200 m source cells per official 1 km parent
A complete 1 km × 1 km cell should contain **25** cells of 200 m × 200 m.

In [ ]:
cells_per_1km = (
    grid_200m.groupby(PARENT_1KM_COL, dropna=False)
    .size()
    .rename('n_200m_cells')
    .reset_index()
)
cells_per_1km['is_complete_25'] = cells_per_1km['n_200m_cells'].eq(25)

print(f'Unique 1 km parent groups: {len(cells_per_1km):,}')
print(f"Complete 25-cell groups: {cells_per_1km['is_complete_25'].sum():,}")
print(f"Groups with count != 25: {(~cells_per_1km['is_complete_25']).sum():,}")
display(
    cells_per_1km['n_200m_cells'].value_counts().sort_index()
    .rename_axis('n_200m_cells')
    .reset_index(name='number_of_1km_parent_groups')
)
display(cells_per_1km[cells_per_1km['n_200m_cells'].ne(25)].sort_values('n_200m_cells'))

## 9. Validate nominal cell size in a metric CRS
EPSG:4674 is geographic. Metric diagnostics therefore use **SIRGAS 2000 / UTM zone 23S (EPSG:31983)**. A nominal 200 m cell should have an area close to 40,000 m².

In [ ]:
METRIC_CRS = 'EPSG:31983'
grid_metric = grid_200m.to_crs(METRIC_CRS)
municipality_metric = municipality.to_crs(METRIC_CRS)

area_m2 = grid_metric.geometry.area
bounds = grid_metric.geometry.bounds
width_m = bounds['maxx'] - bounds['minx']
height_m = bounds['maxy'] - bounds['miny']

print('200 m cell area statistics (m²):')
display(area_m2.describe(percentiles=[0.01, 0.05, 0.50, 0.95, 0.99]).to_frame('area_m2'))
print('Approximate bounding-box dimensions (m):')
display(pd.DataFrame({
    'width_m': width_m.describe(percentiles=[0.01, 0.05, 0.50, 0.95, 0.99]),
    'height_m': height_m.describe(percentiles=[0.01, 0.05, 0.50, 0.95, 0.99]),
}))

EXPECTED_200M_AREA_M2 = 40_000.0
AREA_TOLERANCE = 0.02
within_expected_area = area_m2.between(
    EXPECTED_200M_AREA_M2 * (1 - AREA_TOLERANCE),
    EXPECTED_200M_AREA_M2 * (1 + AREA_TOLERANCE),
)
print('Cells within ±2% of 40,000 m²:', f'{within_expected_area.sum():,} / {len(grid_metric):,}')

## 10. Validate geometric coherence of complete 1 km parent groups
This dissolve is performed **in memory only** for quality control. No file is written.

In [ ]:
validation_1km = (
    grid_metric[[PARENT_1KM_COL, grid_metric.geometry.name]]
    .dissolve(by=PARENT_1KM_COL, as_index=False)
)
validation_1km['area_m2'] = validation_1km.geometry.area
validation_1km = validation_1km.merge(
    cells_per_1km[[PARENT_1KM_COL, 'n_200m_cells', 'is_complete_25']],
    on=PARENT_1KM_COL, how='left'
)
complete_1km = validation_1km[validation_1km['is_complete_25']].copy()
complete_1km['area_within_tolerance'] = complete_1km['area_m2'].between(980_000, 1_020_000)

display(complete_1km['area_m2'].describe(percentiles=[0.01, 0.05, 0.50, 0.95, 0.99]).to_frame('area_m2'))
print('Complete parent groups near 1 km²:', f"{complete_1km['area_within_tolerance'].sum():,} / {len(complete_1km):,}")

## 11. Diagnose the spatial relationship with Petrópolis

In [ ]:
municipality_union = municipality_metric.geometry.union_all()
within = grid_metric.geometry.within(municipality_union)
intersects = grid_metric.geometry.intersects(municipality_union)
relation = np.select(
    [within, intersects],
    ['within', 'boundary_intersection'],
    default='outside',
)
display(
    pd.Series(relation, name='spatial_relation')
    .value_counts()
    .rename_axis('spatial_relation')
    .reset_index(name='n_200m_cells')
)

## 12. Quick diagnostic maps
These figures are for quality control only, not for publication.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 10))
grid_200m.boundary.plot(ax=ax, linewidth=0.15)
municipality.boundary.plot(ax=ax, linewidth=1.2)
ax.set_title('IBGE 200 m grid and Petrópolis municipal boundary')
ax.set_axis_off()
plt.show()

incomplete_ids = set(cells_per_1km.loc[cells_per_1km['n_200m_cells'].ne(25), PARENT_1KM_COL].dropna())
grid_status = grid_200m[[PARENT_1KM_COL, grid_200m.geometry.name]].copy()
grid_status['parent_status'] = np.where(
    grid_status[PARENT_1KM_COL].isin(incomplete_ids),
    'source_count_not_25', 'complete_25'
)
fig, ax = plt.subplots(figsize=(10, 10))
grid_status.plot(column='parent_status', categorical=True, legend=True, ax=ax, linewidth=0)
municipality.boundary.plot(ax=ax, linewidth=1.2)
ax.set_title('200 m cells classified by 1 km parent-group completeness')
ax.set_axis_off()
plt.show()

## 13. PASS / WARNING / FAIL summary

In [ ]:
def status(condition, warning=False):
    if condition:
        return 'PASS'
    return 'WARNING' if warning else 'FAIL'

share_nominal_200m = float(within_expected_area.mean()) if len(within_expected_area) else 0.0
share_nominal_1km = float(complete_1km['area_within_tolerance'].mean()) if len(complete_1km) else 0.0

diagnostics = [
    ['Municipality CRS is defined', status(municipality.crs is not None), str(municipality.crs)],
    ['Grid CRS is defined', status(grid_200m.crs is not None), str(grid_200m.crs)],
    ['Both datasets use the same CRS', status(municipality.crs == grid_200m.crs), str(municipality.crs == grid_200m.crs)],
    ['Municipality resolves to EPSG:4674', status(municipality_epsg == 4674), str(municipality_epsg)],
    ['Grid resolves to EPSG:4674', status(grid_epsg == 4674), str(grid_epsg)],
    ['200 m identifier is unique', status(id_series.is_unique), f'{id_series.nunique(dropna=True):,} unique IDs'],
    ['No null 200 m identifiers', status(id_series.notna().all()), f'{id_series.isna().sum():,} null IDs'],
    ['No null 1 km parent identifiers', status(parent_series.notna().all()), f'{parent_series.isna().sum():,} null parent IDs'],
    ['No 1 km parent contains more than 25 source cells', status(cells_per_1km['n_200m_cells'].max() <= 25), f"maximum = {cells_per_1km['n_200m_cells'].max()}"],
    ['All grid geometries are valid', status(grid_geom_diag['invalid_geometries'] == 0), f"{grid_geom_diag['invalid_geometries']:,} invalid"],
    ['All grid geometries are non-empty', status(grid_geom_diag['empty_geometries'] == 0), f"{grid_geom_diag['empty_geometries']:,} empty"],
    ['200 m cells are approximately 40,000 m²', status(share_nominal_200m >= 0.99, share_nominal_200m >= 0.90), f'{share_nominal_200m:.2%} within ±2%'],
    ['Complete 25-cell parents are approximately 1 km²', status(share_nominal_1km >= 0.99, share_nominal_1km >= 0.90), f'{share_nominal_1km:.2%} within ±2%'],
]
diagnostic_table = pd.DataFrame(diagnostics, columns=['check', 'status', 'value'])
display(diagnostic_table)
display(diagnostic_table['status'].value_counts().reindex(['PASS', 'WARNING', 'FAIL'], fill_value=0).to_frame('count'))

## 14. Decision rule for Notebook 02

Proceed to the 200 m → 1 km aggregation only after confirming the diagnostic summary. The intended aggregation method is based on the **official IBGE 1 km parent identifier**:

```python
grid_1km = grid_200m.dissolve(
    by=PARENT_1KM_COL,
    as_index=False,
)
```

Notebook 02 should preserve the official 1 km identifier, the number of 200 m source cells, a source-group completeness flag, the original 1 km geometry, and the fraction of each 1 km cell intersecting Petrópolis.

No source file is modified by this notebook.